# 0. Extract of CBS Kerncijfers wijken en buurten 2024

Three of our variables come from CBS *Kerncijfers wijken en buurten 2024* (table 85984NED): education level, population density and the share of apartments. The full CBS download is very large, because it contains every district and neighbourhood in the Netherlands. We only need the 342 municipalities and five variables.

In this notebook we make a small table with **one row per municipality**, which the other notebooks use.

**Input** (folder `data/raw/`):
- `kwb2024.zip`: the full CBS download (CSV), with the files `Observations.csv` (all values) and `WijkenEnBuurtenCodes.csv` (the names that belong to the codes)

**Output** (folder `data/raw/`):
- `kwb2024_per_gemeente.csv`: one row per municipality with the CBS municipality code, the name and the five variables

The extract is saved next to the other input files in `data/raw/`, so notebooks 2.2, 5 and 6 can read it from there. Run this notebook before those notebooks.

In [ ]:
import zipfile
import pandas as pd

RAW = "../data/raw/"

## Step 1. Open the CBS download

The CBS download is a zip file with several CSV files in it. With `zipfile` we can read a file from the zip without unpacking it first. We first print which files are in the zip.

In [ ]:
zip_file = zipfile.ZipFile(RAW + "kwb2024.zip")

for name in zip_file.namelist():
    print(name)

We need two of these files. Depending on how the zip was made, they can sit in a subfolder, so we look for the file name at the end of each path.

In [ ]:
for name in zip_file.namelist():
    if name.endswith("Observations.csv"):
        observations_file = name
    if name.endswith("WijkenEnBuurtenCodes.csv"):
        codes_file = name

print("Values:", observations_file)
print("Names: ", codes_file)

## Step 2. Load all values

`Observations.csv` is a long table: every row is one value for one area and one variable. The most important columns are:
- `WijkenEnBuurten`: the code of the area. Municipalities start with `GM`, districts with `WK` and neighbourhoods with `BU`;
- `Measure`: the code of the variable;
- `Value`: the value.

We read everything as text (`dtype=str`), so that codes such as `GM0014` keep their leading zeros. We turn the values into numbers later.

In [ ]:
obs = pd.read_csv(zip_file.open(observations_file), sep=";", dtype=str)

print("Number of rows:", len(obs))
obs.head()

## Step 3. Keep only municipalities and the five variables

The CBS codes of the variables we need are below. We use this dictionary twice: to select the rows, and to give the columns readable names.

In [ ]:
KWB_MEASURES = {
    "M000100": "bevolkingsdichtheid",   # inhabitants per km² of land
    "ZW10340": "meergezinswoning_pct",  # apartments as % of all dwellings
    "2018700": "opl_laag",              # persons 15-75 with a low education level
    "2018740": "opl_middel",            # persons 15-75 with a middle education level
    "2018790": "opl_hoog",              # persons 15-75 with a high education level (hbo, wo)
}

# Only municipalities
obs = obs[obs["WijkenEnBuurten"].str.startswith("GM")]

# Only the five variables we need
obs = obs[obs["Measure"].isin(KWB_MEASURES)]

print("Number of rows left:", len(obs))

## Step 4. One row per municipality

Now every municipality still has five rows, one per variable. With `pivot` we turn this into a table with one row per municipality and one column per variable. Then we replace the CBS codes in the column names with our own names, and turn the text into numbers. Values that CBS does not publish become `NaN` (missing).

In [ ]:
extract = obs.pivot(index="WijkenEnBuurten", columns="Measure", values="Value")
extract = extract.rename(columns=KWB_MEASURES)
extract = extract.apply(pd.to_numeric, errors="coerce")

extract.head()

## Step 5. Add the municipality names

The table only has codes such as `GM0014`. The names are in `WijkenEnBuurtenCodes.csv`: the column `Identifier` has the code and the column `Title` has the name. We use `loc` to look up the name for each code, and put it in a new column at the front.

In [ ]:
codes = pd.read_csv(zip_file.open(codes_file), sep=";", dtype=str)
codes = codes.set_index("Identifier")

extract.insert(0, "gemeente", codes.loc[extract.index, "Title"].str.strip().values)
extract.index.name = "gemeentecode"
extract = extract.reset_index()

print("Number of municipalities:", len(extract))
extract.head()

## Step 6. Check and save

We check that there are 342 municipalities and see which values are missing. Then we save the table.

In [ ]:
missing = extract.isna().sum()
print(missing[missing > 0])

extract[extract.isna().any(axis=1)]

In [ ]:
extract.to_csv(RAW + "kwb2024_per_gemeente.csv", index=False)
print("Saved:", RAW + "kwb2024_per_gemeente.csv")